In [2]:
import os

In [3]:
%pwd

'd:\\Hate-Speech-Classifier\\notebook'

In [4]:
os.chdir("../")

In [5]:
%pwd

'd:\\Hate-Speech-Classifier'

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [12]:
from src.constants import *
from src.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH):
        # Load the configuration from a YAML file
        self.config = read_yaml(config_filepath)
        
        # Create the root directory for artifacts if it doesn't exist
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        # Retrieve the data ingestion configuration section
        ingestion_config = self.config.data_ingestion
        
        # Ensure the root directory for data ingestion exists
        create_directories([ingestion_config.root_dir])

        # Create and return a DataIngestionConfig object
        return DataIngestionConfig(
            root_dir=ingestion_config.root_dir,
            source_URL=ingestion_config.source_URL,
            local_data_file=ingestion_config.local_data_file,
            unzip_dir=ingestion_config.unzip_dir
        )


In [14]:
import os
import urllib.request as request
import zipfile
from src import logger
from src.utils.common import get_size

In [15]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [16]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2024-11-26 14:22:46,858: INFO: common: yaml file: config\config.yaml loaded successfully]
[2024-11-26 14:22:46,898: INFO: common: Created directory at: artifacts]
[2024-11-26 14:22:46,898: INFO: common: Created directory at: artifacts/data_ingestion]
[2024-11-26 14:22:58,055: INFO: 2280763589: artifacts/data_ingestion/dataset.zip download! with following info: 
Connection: close
Content-Length: 2323493
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "6b132494e803a89a55c4fe77ae8e0adedbc79c1ec333c191ed9e8096a0ae4de1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 1B87:28D56:4ECB9F:522D06:6745938D
Accept-Ranges: bytes
Date: Tue, 26 Nov 2024 09:23:26 GMT
Via: 1.1 varnish
X-Served-By: cache-fjr990030-FJR
X-Cache: MISS
X-Cache-Hits: 0
X-Timer: S1732613006.958907,VS0,VE472
Vary: Authorization,Acc